
# IoTShield — 07: Problem formülasyonu
**SE3513 · WP-D3 · ASEADOS-SDN-IoT**

Soru: Bozuk satırlardaki bilgi eksikliğini veri tarafında düzeltemiyorsak (05_overlap), problemi farklı kurarak skor değişiyor mu? Aynı veri, aynı oturum gruplu katlar (`get_folds`), farklı kurgular:
- **D3-0** referans: LightGBM, F_timing, 5 sınıf (E0 ile aynı)
- **11.1** Lojistik regresyon: varsayılan, One-vs-Rest, One-vs-One
- **11.2** PCA (%95 varyans) + lojistik regresyon
- **11.3** Alt küme bazlı: sağlam satırlar ve bozuk satırlar için ayrı modeller
- **11.4** Hiyerarşik: önce Benign / Attack, sonra saldırı tipi
- **11.5** Bot + DDoS birleşik, 4 sınıf (5 sınıfla karşılaştırılmaz)

Tüm sonuçlar `../reports/results_formulation.csv`'ye aynı sütun formatında yazılır. Tek bir sklearn Pipeline olan deneyler `run_cv` ile; 11.3–11.5 kendi döngüyle, ama katlar `get_folds`'tan ve metrikler `src/cv.py`'deki yardımcılarla (Kural 2 istisnası).



## 0. Kurulum


In [1]:

import os
import time
import sys
import numpy as np, pandas as pd

sys.path.append('..')
from src.config import DATA_PATH, RANDOM_STATE, F_TIMING, F_FULL
from src.preprocessing import clean
from src.cv import (get_folds, print_fold_table, run_cv, drop_conflicts, compute_metrics,
                    subset_macro_f1, make_confusion, folds_are_grouped, append_row, CLASSES)
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.decomposition import PCA

pd.set_option('display.max_columns', 30)
FORMULATION_CSV = '../reports/results_formulation.csv'
SHOW_COLS = ['experiment', 'model', 'feature_set', 'macro_f1_mean', 'macro_f1_std', 'macro_f1_clean',
             'macro_f1_broken', 'recall_Dos', 'recall_DDoS', 'recall_Bot', 'recall_Prob', 'bot_ddos_confusion', 'note']



## 1. Veri ve katlar
`clean()` verisi; katlar bir kez alınır ve bütün deneylerde aynı katlar kullanılır. Sonuç dosyası her çalıştırmada sıfırdan yazılır (`run_cv` satır ekler; tekrar çalıştırmada çift satır olmasın).


In [2]:

data = clean(pd.read_csv(DATA_PATH, low_memory=False))
print('veri:', data.shape)

if os.path.exists(FORMULATION_CSV):
    os.remove(FORMULATION_CSV)

folds = get_folds(data)
fold_table = print_fold_table(data, folds)
results = []


clean: dropped 70 rows with inf/NaN
clean: clipped negative values to 0 in 427 rows


clean: removed 29930 duplicate rows (features + label)
clean: final shape (427044, 63)
veri: (427044, 63)


Test kısmındaki satır sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign   76910   79965   76872
Bot       1757    2585    4615
DDoS     23953   10599   15717
Dos      16482   16711   93313
Prob      2803    2316    2446
Test kısmındaki oturum sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign       9       3      11
Bot          1       1       1
DDoS         1       1       2
Dos         19       3       1
Prob         1       1       4



## 2. D3-0 referans: LightGBM, F_timing, 5 sınıf
WP-B1'deki E0 ve 01_EDA Bölüm 9 ile aynı model, aynı özellikler, aynı katlar. Sonuç **0,418 ± 0,119** olmalı; bu aynı zamanda `run_cv`'nin tekrarlanabilirlik kontrolü.


In [3]:

EXPECTED_D3_0 = (0.418, 0.119)    # 01_EDA section 9, F_timing grouped

lgbm_pipe = Pipeline([('model', LGBMClassifier(random_state=RANDOM_STATE, verbose=-1))])
row, cm, oof = run_cv(lgbm_pipe, data, F_TIMING, FORMULATION_CSV, experiment='D3-0', sampler='none',
                      feature_set='F_timing', folds=folds, note='reference (same as E0)')
results.append(row)
display(cm)

assert round(row['macro_f1_mean'], 3) == EXPECTED_D3_0[0], row['macro_f1_mean']
assert round(row['macro_f1_std'], 3) == EXPECTED_D3_0[1], row['macro_f1_std']
print('D3-0 = 01_EDA Bölüm 9 ile aynı: tekrarlanabilirlik kontrolü geçti')


Kat 0


  eğitim: 301069 satır (çelişki atılan: 4070 ) | test: 121905 | fit: 2.7 sn
Kat 1


  eğitim: 309862 satır (çelişki atılan: 5006 ) | test: 112176 | fit: 3.8 sn
Kat 2


  eğitim: 230078 satır (çelişki atılan: 4003 ) | test: 192963 | fit: 4.9 sn


Kat F1'leri: [0.4262 0.5334 0.2952] → 0.4183 ± 0.1193
CSV'ye eklendi: ../reports/results_formulation.csv


,Benign,Dos,DDoS,Bot,Prob
Benign,212974,4617,14828,285,1043
Dos,102595,15001,7610,46,1254
DDoS,9591,3316,35820,762,780
Bot,911,156,7559,276,55
Prob,2388,328,2662,109,2078


D3-0 = 01_EDA Bölüm 9 ile aynı: tekrarlanabilirlik kontrolü geçti



## 3. (11.1) Lojistik regresyon: varsayılan, One-vs-Rest, One-vs-One
Pipeline: `log1p → StandardScaler → LogisticRegression(max_iter=1000)`, F_timing. Ölçekleme ve log Pipeline içinde, yani sadece eğitim katında fit edilir (Kural 4).
- **varsayılan:** tek bir çok sınıflı (multinomial) model
- **OneVsRest:** her sınıf için "bu sınıf mı, diğerleri mi" modeli (5 model)
- **OneVsOne:** her sınıf çifti için bir model (10 model); biri sadece Bot ile DDoS'u ayırmaya odaklanır

Not (Kural 5 istisnası): deney modeli normalde LightGBM; 11.1–11.2 plan gereği lojistik regresyon (hocanın kâğıdındaki OvR / OvO ve PCA adımları). Bu satırlar D3-0 ile karşılaştırılırken model farkı da hesaba katılmalı.


In [4]:

LR_MAX_ITER = 1000


def lr_pipeline(classifier):
    """log1p -> StandardScaler -> classifier, inside an imblearn Pipeline (fit only on the training fold)."""
    return Pipeline([('log', FunctionTransformer(np.log1p)),
                     ('scale', StandardScaler()),
                     ('model', classifier)])


lr_variants = [
    ('D3-11.1-default', LogisticRegression(max_iter=LR_MAX_ITER)),
    ('D3-11.1-OvR', OneVsRestClassifier(LogisticRegression(max_iter=LR_MAX_ITER))),
    ('D3-11.1-OvO', OneVsOneClassifier(LogisticRegression(max_iter=LR_MAX_ITER))),
]
for experiment, classifier in lr_variants:
    print(f'\n=== {experiment} ===')
    row, cm, oof = run_cv(lr_pipeline(classifier), data, F_TIMING, FORMULATION_CSV, experiment=experiment,
                          sampler='none', feature_set='F_timing', folds=folds, note='log1p+StandardScaler+LR')
    results.append(row)

display(pd.DataFrame(results)[SHOW_COLS].round(3))



=== D3-11.1-default ===
Kat 0


  eğitim: 301069 satır (çelişki atılan: 4070 ) | test: 121905 | fit: 29.8 sn


Kat 1


  eğitim: 309862 satır (çelişki atılan: 5006 ) | test: 112176 | fit: 20.4 sn


Kat 2


  eğitim: 230078 satır (çelişki atılan: 4003 ) | test: 192963 | fit: 16.3 sn


... (9 outputs left out)

  eğitim: 309862 satır (çelişki atılan: 5006 ) | test: 112176 | fit: 14.3 sn
Kat 2


  eğitim: 230078 satır (çelişki atılan: 4003 ) | test: 192963 | fit: 14.8 sn


Kat F1'leri: [0.3256 0.4107 0.2781] → 0.3381 ± 0.0672
CSV'ye eklendi: ../reports/results_formulation.csv


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Dos,recall_DDoS,recall_Bot,recall_Prob,bot_ddos_confusion,note
0,D3-0,LGBMClassifier,F_timing,0.418,0.119,0.512,0.324,0.119,0.713,0.031,0.275,0.140,reference (same as E0)
1,D3-11.1-default,LogisticRegression,F_timing,0.352,0.057,0.581,0.254,0.197,0.717,0.001,0.062,0.135,log1p+StandardScaler+LR
2,D3-11.1-OvR,OneVsRestClassifier,F_timing,0.353,0.043,0.639,0.234,0.269,0.730,0.001,0.033,0.132,log1p+StandardScaler+LR
3,D3-11.1-OvO,OneVsOneClassifier,F_timing,0.338,0.067,0.512,0.243,0.118,0.720,0.000,0.053,0.134,log1p+StandardScaler+LR



**Bulgu (11.1):**
- Üç LR versiyonu birbirine yakın: varsayılan **0,352 ± 0,057**, OvR **0,353 ± 0,043**, OvO **0,338 ± 0,067**; hepsi D3-0'ın (LightGBM, 0,418 ± 0,119) altında.
- OvO'daki "sadece Bot vs DDoS" modeli Bot'u ayırmaya yetmiyor: Bot recall'u OvO'da 0,000 (varsayılan 0,001, OvR 0,001). Bu, 05_overlap'taki bulguyla uyumlu: bozuk satırlarda iki sınıfı ayıracak bilgi az.
- LR sağlam alt kümede LightGBM'den yüksek (0,581 / 0,639 / 0,512'ye karşı 0,512), bozuk alt kümede düşük (0,254 / 0,234 / 0,243'e karşı 0,324). Fark çoğunlukla Dos recall'undan (OvR 0,269'a karşı 0,119) ve Prob / Bot'tan geliyor.



## 4. (11.2) PCA + lojistik regresyon
Pipeline: `log1p → StandardScaler → PCA(n_components=0.95) → LogisticRegression`. PCA, varyansın %95'ini açıklayan en az bileşen sayısını eğitim katında kendisi seçer. Skor 11.1 (varsayılan LR) ile karşılaştırılır: tek fark PCA adımı.


In [5]:

PCA_VARIANCE = 0.95

pca_pipe = Pipeline([('log', FunctionTransformer(np.log1p)),
                     ('scale', StandardScaler()),
                     ('pca', PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)),
                     ('model', LogisticRegression(max_iter=LR_MAX_ITER))])
print('=== D3-11.2-PCA ===')
row, cm, oof = run_cv(pca_pipe, data, F_TIMING, FORMULATION_CSV, experiment='D3-11.2-PCA', sampler='none',
                      feature_set='F_timing', folds=folds, note='log1p+StandardScaler+PCA(0.95)+LR')
results.append(row)


=== D3-11.2-PCA ===
Kat 0


  eğitim: 301069 satır (çelişki atılan: 4070 ) | test: 121905 | fit: 7.2 sn
Kat 1


  eğitim: 309862 satır (çelişki atılan: 5006 ) | test: 112176 | fit: 5.5 sn


Kat 2


  eğitim: 230078 satır (çelişki atılan: 4003 ) | test: 192963 | fit: 5.2 sn


Kat F1'leri: [0.3412 0.4129 0.3289] → 0.361 ± 0.0454
CSV'ye eklendi: ../reports/results_formulation.csv



Bileşen sayısı: `run_cv` eğittiği modeli döndürmediği için, her eğitim katında aynı ön işleme (log1p → ölçekleme → PCA) ayrıca fit edilip seçilen bileşen sayısı okunur. Eğitim satırları `run_cv` ile aynı olsun diye önce `drop_conflicts` uygulanır. Bu sadece sayım; skor hesaplanmıyor.


In [6]:

component_rows = []
for fold_no, (train_idx, test_idx) in enumerate(folds):
    train = drop_conflicts(data.iloc[train_idx], F_TIMING, 'Label')
    preprocess = Pipeline([('log', FunctionTransformer(np.log1p)), ('scale', StandardScaler()),
                           ('pca', PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE))])
    preprocess.fit(train[F_TIMING])
    pca_step = preprocess.named_steps['pca']
    component_rows.append({'kat': fold_no, 'bileşen_sayısı': pca_step.n_components_,
                           'açıklanan_varyans': round(pca_step.explained_variance_ratio_.sum(), 3)})
display(pd.DataFrame(component_rows).set_index('kat'))
print(f'F_timing sütun sayısı: {len(F_TIMING)}')

display(pd.DataFrame(results)[SHOW_COLS].round(3))


,bileşen_sayısı,açıklanan_varyans
kat,,
0,7,0.952
1,7,0.950
2,7,0.952


F_timing sütun sayısı: 15


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Dos,recall_DDoS,recall_Bot,recall_Prob,bot_ddos_confusion,note
0,D3-0,LGBMClassifier,F_timing,0.418,0.119,0.512,0.324,0.119,0.713,0.031,0.275,0.140,reference (same as E0)
1,D3-11.1-default,LogisticRegression,F_timing,0.352,0.057,0.581,0.254,0.197,0.717,0.001,0.062,0.135,log1p+StandardScaler+LR
2,D3-11.1-OvR,OneVsRestClassifier,F_timing,0.353,0.043,0.639,0.234,0.269,0.730,0.001,0.033,0.132,log1p+StandardScaler+LR
3,D3-11.1-OvO,OneVsOneClassifier,F_timing,0.338,0.067,0.512,0.243,0.118,0.720,0.000,0.053,0.134,log1p+StandardScaler+LR
4,D3-11.2-PCA,LogisticRegression,F_timing,0.361,0.045,0.665,0.209,0.314,0.809,0.000,0.033,0.113,log1p+StandardScaler+PCA(0.95)+LR



**Bulgu (11.2):**
- PCA her eğitim katında **15 sütundan 7 bileşen** seçiyor (açıklanan varyans %95,0–95,2).
- Skor **0,361 ± 0,045**; PCA'sız LR'den (0,352 ± 0,057) +0,009, katlar arası std'nin içinde. Bilgiyi 7 boyuta sıkıştırmak skoru düşürmüyor; bu da F_timing sütunlarının büyük ölçüde aynı bilgiyi taşıdığıyla (05_overlap 9.1) uyumlu.
- Sağlam alt kümede en yüksek LR skoru (0,665), bozuk alt kümede en düşük (0,209); Bot recall'u 0,000. PCA, LightGBM referansının (0,418) altında kalıyor.



## 5. Kendi CV döngüsü için ortak yardımcılar (11.3–11.5)
11.3–11.5 tek bir sklearn Pipeline değil, bu yüzden `run_cv`'ye verilemez. Aynı katlar (`folds`) üzerinde döngüyü kendimiz yazıyoruz; ama ölçüm `run_cv` ile birebir aynı olsun diye:
- kat F1'i `subset_macro_f1`, OOF metrikleri `compute_metrics` ile,
- CSV satırı `run_cv`'nin sütunlarıyla aynı anahtarlarla, `append_row` ile yazılır.


In [7]:

def lgbm():
    """The fixed experiment model (CLAUDE.md rule 5)."""
    return LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)


def run_custom_cv(fold_function, target, labels):
    """Runs fold_function(train, test) on every fold of `folds` and collects out-of-fold predictions.
    Returns (oof predictions, per-fold macro F1 list, per-fold fit seconds, total predict seconds)."""
    y_true = data[target].to_numpy()
    oof = np.empty(len(data), dtype=object)
    fold_f1s, fit_times, predict_seconds_total = [], [], 0.0
    for fold_no, (train_idx, test_idx) in enumerate(folds):
        print('Kat', fold_no)
        predictions, fit_seconds, predict_seconds = fold_function(data.iloc[train_idx], data.iloc[test_idx])
        oof[test_idx] = predictions
        fold_f1s.append(subset_macro_f1(y_true[test_idx], predictions, labels))   # same as run_cv
        fit_times.append(fit_seconds)
        predict_seconds_total += predict_seconds
    print("Kat F1'leri:", np.round(fold_f1s, 4), '→', round(np.mean(fold_f1s), 4), '±', round(np.std(fold_f1s, ddof=1), 4))
    return oof, fold_f1s, fit_times, predict_seconds_total


def make_result_row(experiment, model_name, feature_set, target, oof, fold_f1s, fit_times, predict_seconds, note):
    """Builds a result row with exactly the same keys and formulas as run_cv (src/cv.py)."""
    y_true = data[target].to_numpy()
    metrics = compute_metrics(y_true, oof, data['broken'])
    row = {'experiment': experiment, 'model': model_name, 'sampler': 'none', 'feature_set': feature_set,
           'cv': 'grouped' if folds_are_grouped(data, folds) else 'random',
           'macro_f1_mean': np.mean(fold_f1s), 'macro_f1_std': np.std(fold_f1s, ddof=1),
           'macro_f1_clean': metrics['macro_f1_clean'], 'macro_f1_broken': metrics['macro_f1_broken']}
    for class_name in CLASSES:
        row['recall_' + class_name] = metrics['recall_' + class_name]
    row['bot_ddos_confusion'] = metrics['bot_ddos_confusion']
    row['fit_time_s'] = np.mean(fit_times)
    row['pred_ms_per_flow'] = predict_seconds / len(data) * 1000
    row['note'] = note
    return row



## 6. (11.3) Alt küme bazlı modeller: sağlam ve bozuk satırlar için ayrı model
- **Sağlam satırlar** (`broken == False`, sadece Benign / Dos): LightGBM, **F_full** (bayt sütunları sağlam satırlarda gerçek değer taşıyor).
- **Bozuk satırlar** (5 sınıf): LightGBM, **F_timing**.
- Tahminde her satır `broken` bayrağına göre ilgili modele gider. Çelişkili satırlar her modelin kendi özellik setinde, sadece eğitim katından atılır (Kural 4).

> ⚠️ **Uyarı: `broken` bayrağı bir kayıt artefaktı.** `Protocol == 0`, CICFlowMeter'ın bazı kayıt oturumlarında başlıkları okuyamamasından geliyor (01_EDA Bölüm 4); trafiğin bir özelliği değil. Bu veri setinde her satır için biliniyor, ama gerçek trafikte (ör. Mininet) böyle bir bayrak olmayabilir ya da başka bir anlama gelebilir. Bu yüzden buradaki yönlendirme **gerçek kullanımda anlamsız olabilir**; 11.3'ün skoru "en iyi model" olarak değil, "iki alt problem ayrı çözülürse ne olur" sorusunun cevabı olarak okunmalı. Sunumda da bunu söyleyeceğiz.


In [8]:

def subset_models_fold(train, test):
    """11.3, one fold: healthy rows -> LGBM on F_FULL (Benign/Dos), broken rows -> LGBM on F_TIMING (5 classes).
    Test rows are routed by the broken flag. Returns (predictions, fit seconds, predict seconds)."""
    healthy_train = drop_conflicts(train[~train['broken']], F_FULL, 'Label')
    broken_train = drop_conflicts(train[train['broken']], F_TIMING, 'Label')
    print('  sağlam eğitim:', len(healthy_train), '| bozuk eğitim:', len(broken_train))

    start = time.perf_counter()
    healthy_model = lgbm().fit(healthy_train[F_FULL], healthy_train['Label'])
    broken_model = lgbm().fit(broken_train[F_TIMING], broken_train['Label'])
    fit_seconds = time.perf_counter() - start

    is_broken = test['broken'].to_numpy()
    predictions = np.empty(len(test), dtype=object)
    start = time.perf_counter()
    predictions[~is_broken] = healthy_model.predict(test[~is_broken][F_FULL])
    predictions[is_broken] = broken_model.predict(test[is_broken][F_TIMING])
    predict_seconds = time.perf_counter() - start
    return predictions, fit_seconds, predict_seconds


print('=== D3-11.3-subset ===')
oof, fold_f1s, fit_times, predict_seconds = run_custom_cv(subset_models_fold, 'Label', CLASSES)
row = make_result_row('D3-11.3-subset', 'LGBMClassifier x2 (routed by broken)', 'F_full (healthy) / F_timing (broken)',
                      'Label', oof, fold_f1s, fit_times, predict_seconds,
                      note='routed by broken flag (recording artifact; may be meaningless on real traffic)')
append_row(row, FORMULATION_CSV)
results.append(row)
display(make_confusion(data['Label'].to_numpy(), oof))
display(pd.DataFrame(results)[SHOW_COLS].round(3))


=== D3-11.3-subset ===
Kat 0


  sağlam eğitim: 232416 | bozuk eğitim: 71723


Kat 1


  sağlam eğitim: 222590 | bozuk eğitim: 91418


Kat 2


  sağlam eğitim: 150178 | bozuk eğitim: 83286


Kat F1'leri: [0.4664 0.5541 0.4918] → 0.5041 ± 0.0451


CSV'ye eklendi: ../reports/results_formulation.csv


,Benign,Dos,DDoS,Bot,Prob
Benign,212954,4809,15090,233,661
Dos,45961,72318,7946,134,147
DDoS,8608,3844,36000,744,1073
Bot,441,501,7619,326,70
Prob,1494,618,2808,104,2541


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Dos,recall_DDoS,recall_Bot,recall_Prob,bot_ddos_confusion,note
0,D3-0,LGBMClassifier,F_timing,0.418,0.119,0.512,0.324,0.119,0.713,0.031,0.275,0.140,reference (same as E0)
1,D3-11.1-default,LogisticRegression,F_timing,0.352,0.057,0.581,0.254,0.197,0.717,0.001,0.062,0.135,log1p+StandardScaler+LR
2,D3-11.1-OvR,OneVsRestClassifier,F_timing,0.353,0.043,0.639,0.234,0.269,0.730,0.001,0.033,0.132,log1p+StandardScaler+LR
3,D3-11.1-OvO,OneVsOneClassifier,F_timing,0.338,0.067,0.512,0.243,0.118,0.720,0.000,0.053,0.134,log1p+StandardScaler+LR
4,D3-11.2-PCA,LogisticRegression,F_timing,0.361,0.045,0.665,0.209,0.314,0.809,0.000,0.033,0.113,log1p+StandardScaler+PCA(0.95)+LR
5,D3-11.3-subset,LGBMClassifier x2 (routed by broken),F_full (healthy) / F_timing (broken),0.504,0.045,0.842,0.343,0.572,0.716,0.036,0.336,0.141,routed by broken flag (recording artifact; may...



**Bulgu (11.3):**
- Alt küme bazlı kurgu **0,504 ± 0,045**; D3-0'dan (0,418 ± 0,119) +0,086 yüksek. Artış sağlam alt kümeden: sağlam F1 0,842 (D3-0: 0,512), Dos recall 0,572 (D3-0: 0,119). Bozuk alt kümede küçük (0,343'e karşı 0,324); Bot recall 0,036.
- Karşılaştırma: 01_EDA Bölüm 9'da aynı katlarda **tek bir F_full modeli** 0,499 ± 0,052 vermişti (sağlam 0,837, bozuk 0,339). Yani artışın neredeyse tamamı sağlam satırlarda bayt sütunlarını (F_full) kullanmaktan geliyor; yönlendirmenin kendisinin katkısı küçük (+0,005).
- Yukarıdaki uyarı geçerli: yönlendirme `broken` bayrağına dayanıyor ve bu bayrak bir kayıt artefaktı. Bu satır "en iyi model" değil, "iki alt problemi ayrı çözersek ne olur" sorusunun cevabı.



## 7. (11.4) Hiyerarşik model: önce Benign / Attack, sonra saldırı tipi
- **1. aşama:** LightGBM, F_timing, hedef `label_binary` (Benign = 0, saldırı = 1). Tüm eğitim satırlarıyla.
- **2. aşama:** LightGBM, F_timing, hedef `Label` (Dos / DDoS / Bot / Prob). Sadece **gerçek** saldırı satırlarıyla eğitilir.
- Tahmin: 1. aşama 0 derse sonuç "Benign"; 1 derse satır 2. aşamaya gider ve saldırı tipi oradan gelir.
- Skor **uçtan uca 5 sınıf** üzerinden: 1. aşamanın hataları (Benign sanılan saldırılar, saldırı sanılan Benign'ler) skora dahil.
- Çelişkili satırlar her aşamada o aşamanın kendi hedefiyle, sadece eğitim katından atılır (1. aşama `label_binary`, 2. aşama `Label`).


In [9]:

def hierarchical_fold(train, test):
    """11.4, one fold: stage 1 = Benign/Attack on all rows, stage 2 = attack type on true attack rows only.
    Returns (5-class predictions, fit seconds, predict seconds)."""
    stage1_train = drop_conflicts(train, F_TIMING, 'label_binary')
    attack_rows = train[train['label_binary'] == 1]
    stage2_train = drop_conflicts(attack_rows, F_TIMING, 'Label')
    print('  1. aşama eğitim:', len(stage1_train), '| 2. aşama eğitim (gerçek saldırılar):', len(stage2_train))

    start = time.perf_counter()
    stage1 = lgbm().fit(stage1_train[F_TIMING], stage1_train['label_binary'])
    stage2 = lgbm().fit(stage2_train[F_TIMING], stage2_train['Label'])
    fit_seconds = time.perf_counter() - start

    start = time.perf_counter()
    is_attack = stage1.predict(test[F_TIMING]) == 1
    predictions = np.full(len(test), 'Benign', dtype=object)
    predictions[is_attack] = stage2.predict(test[is_attack][F_TIMING])
    predict_seconds = time.perf_counter() - start
    return predictions, fit_seconds, predict_seconds


print('=== D3-11.4-hierarchical ===')
oof, fold_f1s, fit_times, predict_seconds = run_custom_cv(hierarchical_fold, 'Label', CLASSES)
row = make_result_row('D3-11.4-hierarchical', 'LGBMClassifier x2 (Benign/Attack -> type)', 'F_timing',
                      'Label', oof, fold_f1s, fit_times, predict_seconds,
                      note='end-to-end 5 classes, stage-1 errors included')
append_row(row, FORMULATION_CSV)
results.append(row)
display(make_confusion(data['Label'].to_numpy(), oof))


=== D3-11.4-hierarchical ===
Kat 0


  1. aşama eğitim: 301235 | 2. aşama eğitim (gerçek saldırılar): 147521


Kat 1


  1. aşama eğitim: 309980 | 2. aşama eğitim (gerçek saldırılar): 160345


Kat 2


  1. aşama eğitim: 230168 | 2. aşama eğitim (gerçek saldırılar): 76784


Kat F1'leri: [0.4396 0.5317 0.2989] → 0.4234 ± 0.1173


CSV'ye eklendi: ../reports/results_formulation.csv


,Benign,Dos,DDoS,Bot,Prob
Benign,206810,7258,18358,246,1075
Dos,98273,17126,9906,54,1147
DDoS,5521,5126,37679,835,1108
Bot,674,213,7746,240,84
Prob,1533,662,2870,88,2412



1. aşamanın payı: uçtan uca tahminden "Benign mi, saldırı mı" kısmı çıkarılıp `compute_metrics` ile ikili skor hesaplanır; ayrıca saldırı satırlarının yüzde kaçı 1. aşamada Benign sanılıp 2. aşamaya hiç ulaşmadı (sınıf bazında).


In [10]:

true_binary = data['label_binary'].to_numpy()
predicted_binary = (oof != 'Benign').astype(int)
stage1_metrics = compute_metrics(true_binary, predicted_binary, data['broken'])
print('1. aşama (ikili) macro F1 — tüm: %.3f | sağlam: %.3f | bozuk: %.3f'
      % (stage1_metrics['macro_f1'], stage1_metrics['macro_f1_clean'], stage1_metrics['macro_f1_broken']))

lost_rows = []
for label in ['Dos', 'DDoS', 'Bot', 'Prob']:
    is_label = data['Label'].to_numpy() == label
    lost_rows.append({'sınıf': label, 'satır': int(is_label.sum()),
                      '1. aşamada Benign sanılan %': round((oof[is_label] == 'Benign').mean() * 100, 1)})
display(pd.DataFrame(lost_rows).set_index('sınıf'))
display(pd.DataFrame(results)[SHOW_COLS].round(3))


1. aşama (ikili) macro F1 — tüm: 0.662 | sağlam: 0.531 | bozuk: 0.640


,satır,1. aşamada Benign sanılan %
sınıf,,
Dos,126506,77.7
DDoS,50269,11.0
Bot,8957,7.5
Prob,7565,20.3


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Dos,recall_DDoS,recall_Bot,recall_Prob,bot_ddos_confusion,note
0,D3-0,LGBMClassifier,F_timing,0.418,0.119,0.512,0.324,0.119,0.713,0.031,0.275,0.140,reference (same as E0)
1,D3-11.1-default,LogisticRegression,F_timing,0.352,0.057,0.581,0.254,0.197,0.717,0.001,0.062,0.135,log1p+StandardScaler+LR
2,D3-11.1-OvR,OneVsRestClassifier,F_timing,0.353,0.043,0.639,0.234,0.269,0.730,0.001,0.033,0.132,log1p+StandardScaler+LR
3,D3-11.1-OvO,OneVsOneClassifier,F_timing,0.338,0.067,0.512,0.243,0.118,0.720,0.000,0.053,0.134,log1p+StandardScaler+LR
4,D3-11.2-PCA,LogisticRegression,F_timing,0.361,0.045,0.665,0.209,0.314,0.809,0.000,0.033,0.113,log1p+StandardScaler+PCA(0.95)+LR
5,D3-11.3-subset,LGBMClassifier x2 (routed by broken),F_full (healthy) / F_timing (broken),0.504,0.045,0.842,0.343,0.572,0.716,0.036,0.336,0.141,routed by broken flag (recording artifact; may...
6,D3-11.4-hierarchical,LGBMClassifier x2 (Benign/Attack -> type),F_timing,0.423,0.117,0.521,0.327,0.135,0.750,0.027,0.319,0.145,"end-to-end 5 classes, stage-1 errors included"



**Bulgu (11.4):**
- Hiyerarşik kurgu uçtan uca **0,423 ± 0,117**; D3-0 (0,418 ± 0,119) ile neredeyse aynı (+0,005, std içinde). Kat skorları da D3-0'a çok yakın (0,440 / 0,532 / 0,299).
- 1. aşama (Benign / Attack) ikili macro F1 0,662. En büyük kayıp Dos'ta: Dos satırlarının **%77,7'si 1. aşamada Benign** sanılıyor ve 2. aşamaya hiç ulaşmıyor (DDoS %11,0, Bot %7,5, Prob %20,3).
- Bot / DDoS sorunu 2. aşamada da sürüyor: 1. aşamayı geçen Bot satırlarının çoğu DDoS tahmin ediliyor (Bot recall 0,027).
- Yani problemi iki aşamaya bölmek, iki ana zorluğu (Dos ↔ Benign ve Bot ↔ DDoS) ayırıyor ama ikisini de çözmüyor; zorluklar aşamalara taşınıyor.



## 8. (11.5) Bot + DDoS birleşik: 4 sınıf
05_overlap'ta bozuk satırlarda Bot ile DDoS'u ayıracak bilginin çok az olduğunu gördük (sadece süre ≈ F_timing, dengeleme ayrımı iyileştirmiyor, kNN komşuları karışık). Bu kurguda iki sınıf tek sınıf sayılır: hedefte **Bot → DDoS** ("DDoS" = DDoS ∪ Bot). Model: LightGBM, F_timing (D3-0 ile tek fark hedef).

**Bu satır 5 sınıflı satırlarla karşılaştırılmaz:** sınıf sayısı farklı ve en zor çift (Bot ↔ DDoS) tanım gereği ortadan kalkıyor. Macro F1 sadece mevcut 4 etiket (Benign, Dos, DDoS, Prob) üzerinden hesaplanır; `recall_Bot` ve `bot_ddos_confusion` bu kurguda tanımsız olduğu için CSV'de boş (NaN) bırakılır.


In [11]:

FOUR_CLASSES = ['Benign', 'Dos', 'DDoS', 'Prob']
data['label_4'] = data['Label'].replace({'Bot': 'DDoS'})
display(data['label_4'].value_counts().reindex(FOUR_CLASSES))


def four_class_fold(train, test):
    """11.5, one fold: LGBM on F_TIMING with target label_4 (Bot merged into DDoS).
    Returns (predictions, fit seconds, predict seconds)."""
    train = drop_conflicts(train, F_TIMING, 'label_4')
    start = time.perf_counter()
    model = lgbm().fit(train[F_TIMING], train['label_4'])
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    predictions = model.predict(test[F_TIMING])
    predict_seconds = time.perf_counter() - start
    return predictions, fit_seconds, predict_seconds


print('=== D3-11.5-4class ===')
oof, fold_f1s, fit_times, predict_seconds = run_custom_cv(four_class_fold, 'label_4', FOUR_CLASSES)
row = make_result_row('D3-11.5-4class', 'LGBMClassifier', 'F_timing', 'label_4', oof, fold_f1s, fit_times,
                      predict_seconds,
                      note='4 sınıf (Bot→DDoS birleşik); 5 sınıflı satırlarla karşılaştırılmaz; '
                           'recall_Bot/bot_ddos_confusion tanımsız')
# Bot does not exist in this target: these two fields are undefined, not 0
row['recall_Bot'] = np.nan
row['bot_ddos_confusion'] = np.nan
append_row(row, FORMULATION_CSV)
results.append(row)
display(make_confusion(data['label_4'].to_numpy(), oof).loc[FOUR_CLASSES, FOUR_CLASSES])


label_4
Benign    233747
Dos       126506
DDoS       59226
Prob        7565
Name: count, dtype: int64

=== D3-11.5-4class ===
Kat 0


Kat 1


Kat 2


Kat F1'leri: [0.5161 0.6842 0.3981] → 0.5328 ± 0.1438


CSV'ye eklendi: ../reports/results_formulation.csv


,Benign,Dos,DDoS,Prob
Benign,212642,4660,15477,968
Dos,101115,16207,7822,1362
DDoS,9871,3425,45118,812
Prob,2384,306,2878,1997



**Bulgu (11.5):**
- 4 sınıflı kurgu **0,533 ± 0,144** (kat 0,516 / 0,684 / 0,398). Bu sayı 5 sınıflı satırlarla karşılaştırılmaz: sınıf sayısı farklı ve Bot ↔ DDoS ayrımı tanım gereği yok.
- Birleşik "DDoS" sınıfının recall'u 0,762 (DDoS ∪ Bot); bozuk alt küme F1'i 0,410.
- Kalan en büyük hata Dos ↔ Benign: Dos recall'u 0,128, Dos satırlarının çoğu Benign tahmin ediliyor (D3-0 ile aynı sorun). Bot ile DDoS'u birleştirmek bu sorunu değiştirmiyor.
- `recall_Bot` ve `bot_ddos_confusion` CSV'de boş (tanımsız).



Tüm deneyler (son satır 4 sınıflı, ayrı okunmalı):


In [12]:

summary = pd.DataFrame(results)[SHOW_COLS]
display(summary.round(3))


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Dos,recall_DDoS,recall_Bot,recall_Prob,bot_ddos_confusion,note
0,D3-0,LGBMClassifier,F_timing,0.418,0.119,0.512,0.324,0.119,0.713,0.031,0.275,0.140,reference (same as E0)
1,D3-11.1-default,LogisticRegression,F_timing,0.352,0.057,0.581,0.254,0.197,0.717,0.001,0.062,0.135,log1p+StandardScaler+LR
2,D3-11.1-OvR,OneVsRestClassifier,F_timing,0.353,0.043,0.639,0.234,0.269,0.730,0.001,0.033,0.132,log1p+StandardScaler+LR
3,D3-11.1-OvO,OneVsOneClassifier,F_timing,0.338,0.067,0.512,0.243,0.118,0.720,0.000,0.053,0.134,log1p+StandardScaler+LR
4,D3-11.2-PCA,LogisticRegression,F_timing,0.361,0.045,0.665,0.209,0.314,0.809,0.000,0.033,0.113,log1p+StandardScaler+PCA(0.95)+LR
5,D3-11.3-subset,LGBMClassifier x2 (routed by broken),F_full (healthy) / F_timing (broken),0.504,0.045,0.842,0.343,0.572,0.716,0.036,0.336,0.141,routed by broken flag (recording artifact; may...
6,D3-11.4-hierarchical,LGBMClassifier x2 (Benign/Attack -> type),F_timing,0.423,0.117,0.521,0.327,0.135,0.750,0.027,0.319,0.145,"end-to-end 5 classes, stage-1 errors included"
7,D3-11.5-4class,LGBMClassifier,F_timing,0.533,0.144,0.522,0.410,0.128,0.762,NaN,0.264,NaN,4 sınıf (Bot→DDoS birleşik); 5 sınıflı satırla...
